In [1]:
!pip install ogx_client

Looking in indexes: https://packages.redhat.com/api/pypi/public-rhai/rhoai/3.5-EA2/cpu-ubi9-test/simple/


In [1]:
from ogx_client import OgxClient
import rich

In [2]:
# Configuration
OGX_CONNECTION_URL = "http://ogxserver-service.llama.svc.cluster.local:8321"
POLICY_FILE = "../data/return-policy.txt"

In [3]:
# Initialize OGX client
client = OgxClient(base_url=OGX_CONNECTION_URL)

In [4]:
# List available models
models = client.models.list()
rich.print(models)

ListModelsResponse(
    data=[
        Model(
            id='vllm-embedding/granite-embeddings',
            created=1781273092,
            owned_by='ogx',
            custom_metadata={
                'model_type': 'embedding',
                'provider_id': 'vllm-embedding',
                'provider_resource_id': 'granite-embeddings',
                'embedding_dimension': 768
            },
            object='model'
        ),
        Model(
            id='vllm-inference/llama-32-3b-instruct',
            created=1781273092,
            owned_by='ogx',
            custom_metadata={
                'model_type': 'llm',
                'provider_id': 'vllm-inference',
                'provider_resource_id': 'llama-32-3b-instruct'
            },
            object='model'
        )
    ],
    object='list'
)

In [12]:
# Checking if any vector db present
rich.print(client.vector_stores.list())
#rich.print(client.vector_stores.delete("vs_99e0329b-d721-4cb9-a07a-c3578d1dfa34"))

SyncOpenAICursorPage[VectorStore](data=[], has_more=False, last_id='', object='list', first_id='')

In [13]:
# Extract LLM and embedding model details
llm_model = next(
    m for m in models.data
    if m.custom_metadata.get("model_type") == "llm"
)

# Using specifically sentence-transformers because customized the config to use this inline model
embedding_model = next(
    m for m in models.data
    if m.custom_metadata.get("model_type") == "embedding" 
)

model_id = llm_model.id
embedding_model_id = embedding_model.id
embedding_dimension = embedding_model.custom_metadata["embedding_dimension"]

print(f"LLM Model: {model_id}")
print(f"Embedding Model: {embedding_model_id}")
print(f"Embedding Dimension: {embedding_dimension}")

LLM Model: vllm-inference/llama-32-3b-instruct
Embedding Model: vllm-embedding/granite-embeddings
Embedding Dimension: 768


### Verify Vector Store files

In [44]:
# Create vector store with remote qdrant
vector_store = client.vector_stores.create(
    name="techmart_policy_store_remote_qdrant",
    extra_body={
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "qdrant-remote",
    },
)

vector_store_id = vector_store.id
print(f"Created vector store: {vector_store_id}")

Created vector store: vs_4c6b1086-c038-431a-932b-c513cfadd9c1


In [15]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_1104730f-068a-4684-8eb4-0328102a7035',
            created_at=1781273225,
            file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1781273225,
            metadata={
                'provider_id': 'qdrant-remote',
                'provider_vector_store_id': 'vs_1104730f-068a-4684-8eb4-0328102a7035',
                'embedding_model': 'vllm-embedding/granite-embeddings',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store_remote_qdrant',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
    object='list',
    first_id='vs_1104730f-068a-4684-8eb4-0328102a7035'
)

In [17]:
# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info.id}")

Uploaded file: file-ffadbac75f974ca989438f58d5a489c3


In [18]:
# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info2 = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info2.id}")

Uploaded file: file-0c15a80946a140198ac6e20fc0e60e8d


In [19]:
# Add file to vector store with chunking strategy
vector_store_file = client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_info.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 400,
            "chunk_overlap_tokens": 100,
        },
    },
)

rich.print(vector_store_file)

VectorStoreFile(
    id='file-ffadbac75f974ca989438f58d5a489c3',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781273257,
    status='completed',
    vector_store_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [20]:
# Add file to vector store with chunking strategy
vector_store_file_2 = client.vector_stores.files.create(
    vector_store_id=vector_store_id,
    file_id=file_info2.id,
    chunking_strategy={
        "type": "static",
        "static": {
            "max_chunk_size_tokens": 400,
            "chunk_overlap_tokens": 100,
        },
    },
)

rich.print(vector_store_file_2)

VectorStoreFile(
    id='file-0c15a80946a140198ac6e20fc0e60e8d',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781273265,
    status='completed',
    vector_store_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [21]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_1104730f-068a-4684-8eb4-0328102a7035',
            created_at=1781273225,
            file_counts=FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1781273225,
            metadata={
                'provider_id': 'qdrant-remote',
                'provider_vector_store_id': 'vs_1104730f-068a-4684-8eb4-0328102a7035',
                'embedding_model': 'vllm-embedding/granite-embeddings',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store_remote_qdrant',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
    object='list',
    first_id='vs_1104730f-068a-4684-8eb4-0328102a7035'
)

In [22]:
# Verify file is completed
files = client.vector_stores.files.list(vector_store_id)
rich.print(files)

SyncOpenAICursorPage[VectorStoreFile](
    data=[
        VectorStoreFile(
            id='file-0c15a80946a140198ac6e20fc0e60e8d',
            chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
                static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
                    chunk_overlap_tokens=100,
                    max_chunk_size_tokens=400
                ),
                type='static'
            ),
            created_at=1781273265,
            status='completed',
            vector_store_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
            attributes={},
            last_error=None,
            object='vector_store.file',
            usage_bytes=0
        ),
        VectorStoreFile(
            id='file-ffadbac75f974ca989438f58d5a489c3',
            chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
                static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
                    chunk_overlap_tokens=100,
                    max_chunk_size_tokens=400
                ),
                type='static'
            ),
            created_at=1781273257,
            status='completed',
            vector_store_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
            attributes={},
            last_error=None,
            object='vector_store.file',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='file-ffadbac75f974ca989438f58d5a489c3',
    object='list',
    first_id='file-0c15a80946a140198ac6e20fc0e60e8d'
)

In [23]:
# Inspects a specific file inside (in our case first file) the store to check if processing is 'completed'
file_metadata = client.vector_stores.files.retrieve(
    vector_store_id=vector_store_id,
    file_id=file_info.id
)
# Use attributes that are standard to VectorStoreFile objects
print(f"File ID: {file_metadata.id}")
print(f"Processing Status: {file_metadata.status}")
rich.print(file_metadata)

File ID: file-ffadbac75f974ca989438f58d5a489c3
Processing Status: completed


VectorStoreFile(
    id='file-ffadbac75f974ca989438f58d5a489c3',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781273257,
    status='completed',
    vector_store_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [24]:
# Call the content method
file_chunks_content = client.vector_stores.files.content(
    vector_store_id=vector_store_id,
    file_id=file_info.id
)

print("--- Extracted File Content ---")
# Loop through the rows/chunks stored inside the data attribute
for item in file_chunks_content.data:
    # Try accessing the text payload (typically under .content, .text, or .chunk)
    if hasattr(item, "content"):
        print(item.content)
    elif hasattr(item, "text"):
        print(item.text)
    else:
        # Fallback if the item itself is a dictionary or direct string
        print(item)

--- Extracted File Content ---
TechMart Return and Refund Policy
Shipping Information:
- Standard Shipping: 3-5 business days (Free on orders over $50)
- Express Shipping: 1-2 business days ($15.99)
- Overnight Shipping: Next business day ($29.99, order before 2 PM EST)
- Orders are processed within 24 hours on business days
- Tracking number sent via email once shipped
Return Time Limits:
- Standard items can be returned within 30 days of delivery
- Electronics must be returned within 15 days of delivery
- Opened software and personalized items cannot be returned
Return Conditions:
Items must be in original condition with original packaging intact. All accessories, manuals, and tags must be included. Items showing signs of use may receive partial refund or be rejected.
How to Return an Item:
1. Log into your TechMart account
2. Go to My Orders and select the order
3. Click Return Item and choose a reason
4. You will receive a return label via email within 24 hours
5. Pack the item sec

In [25]:
# Update file attributes/metadata inside the vector store
updated_file = client.vector_stores.files.update(
    file_id=file_info.id,
    vector_store_id=vector_store_id,
    attributes={
        "department": "return-support",
        "last_updated_by": "admin",
        "version": "2.0"
    }
)

print(f"File updated! New attribute state: {updated_file}")

File updated! New attribute state: VectorStoreFile(id='file-ffadbac75f974ca989438f58d5a489c3', chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(chunk_overlap_tokens=100, max_chunk_size_tokens=400), type='static'), created_at=1781273257, status='completed', vector_store_id='vs_1104730f-068a-4684-8eb4-0328102a7035', attributes={'department': 'return-support', 'last_updated_by': 'admin', 'version': '2.0'}, last_error=None, object='vector_store.file', usage_bytes=0)


In [26]:
# Check updated attributes
file_metadata = client.vector_stores.files.retrieve(
    vector_store_id=vector_store_id,
    file_id=file_info.id
)
# Use attributes that are standard to VectorStoreFile objects
print(f"File ID: {file_metadata.id}")
rich.print(file_metadata)

File ID: file-ffadbac75f974ca989438f58d5a489c3


VectorStoreFile(
    id='file-ffadbac75f974ca989438f58d5a489c3',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781273257,
    status='completed',
    vector_store_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
    attributes={'department': 'return-support', 'last_updated_by': 'admin', 'version': '2.0'},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [27]:
# Check attributes of second file
file_metadata = client.vector_stores.files.retrieve(
    vector_store_id=vector_store_id,
    file_id=file_info2.id
)
# Use attributes that are standard to VectorStoreFile objects
print(f"File ID: {file_metadata.id}")
rich.print(file_metadata)

File ID: file-0c15a80946a140198ac6e20fc0e60e8d


VectorStoreFile(
    id='file-0c15a80946a140198ac6e20fc0e60e8d',
    chunking_strategy=ChunkingStrategyVectorStoreChunkingStrategyStatic(
        static=ChunkingStrategyVectorStoreChunkingStrategyStaticStatic(
            chunk_overlap_tokens=100,
            max_chunk_size_tokens=400
        ),
        type='static'
    ),
    created_at=1781273265,
    status='completed',
    vector_store_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
    attributes={},
    last_error=None,
    object='vector_store.file',
    usage_bytes=0
)

In [28]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_1104730f-068a-4684-8eb4-0328102a7035',
            created_at=1781273225,
            file_counts=FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1781273225,
            metadata={
                'provider_id': 'qdrant-remote',
                'provider_vector_store_id': 'vs_1104730f-068a-4684-8eb4-0328102a7035',
                'embedding_model': 'vllm-embedding/granite-embeddings',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store_remote_qdrant',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
    object='list',
    first_id='vs_1104730f-068a-4684-8eb4-0328102a7035'
)

In [42]:
file_metadata = client.vector_stores.files.delete(
    vector_store_id=vector_store_id,
    file_id=file_info.id
)

BadRequestError: Error code: 400 - {'error': {'message': 'File file-ffadbac75f974ca989438f58d5a489c3 not found in vector store vs_0d1bebf0-fd5d-4aed-91f3-cee8dfeff415'}}

In [30]:
file_metadata = client.vector_stores.files.delete(
    vector_store_id=vector_store_id,
    file_id=file_info2.id
)

In [31]:
rich.print(client.vector_stores.files.list(vector_store_id=vector_store_id))

SyncOpenAICursorPage[VectorStoreFile](data=[], has_more=False, last_id='', object='list', first_id='')

In [32]:
rich.print(client.vector_stores.list())

SyncOpenAICursorPage[VectorStore](
    data=[
        VectorStore(
            id='vs_1104730f-068a-4684-8eb4-0328102a7035',
            created_at=1781273225,
            file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=0, total=0),
            status='completed',
            expires_after=None,
            expires_at=None,
            last_active_at=1781273225,
            metadata={
                'provider_id': 'qdrant-remote',
                'provider_vector_store_id': 'vs_1104730f-068a-4684-8eb4-0328102a7035',
                'embedding_model': 'vllm-embedding/granite-embeddings',
                'embedding_dimension': '768'
            },
            name='techmart_policy_store_remote_qdrant',
            object='vector_store',
            usage_bytes=0
        )
    ],
    has_more=False,
    last_id='vs_1104730f-068a-4684-8eb4-0328102a7035',
    object='list',
    first_id='vs_1104730f-068a-4684-8eb4-0328102a7035'
)

### Verify Vector Store files batch

In [45]:
# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info1 = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info1.id}")

# Upload return policy document
with open(POLICY_FILE, "rb") as f:
    file_info2 = client.files.create(
        file=("return-policy.txt", f),
        purpose="assistants",
    )

print(f"Uploaded file: {file_info2.id}")

Uploaded file: file-b4fc6750d8304dc798ab0d317c332c4a
Uploaded file: file-879c85a415534667a0b595b32175de90


In [46]:
batch = client.vector_stores.file_batches.create(
    vector_store_id=vector_store_id,
    file_ids=[file_info1.id, file_info2.id],
)
print(batch)

VectorStoreFileBatches(id='batch_2094edf6-aa4d-4419-b08c-004e75f7d5bc', created_at=1781273651, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=2, total=2), status='in_progress', vector_store_id='vs_4c6b1086-c038-431a-932b-c513cfadd9c1', object='vector_store.files_batch')


In [47]:
batch = client.vector_stores.file_batches.retrieve(
    vector_store_id=vector_store_id,
    batch_id=batch.id,
)
print(batch)

VectorStoreFileBatches(id='batch_2094edf6-aa4d-4419-b08c-004e75f7d5bc', created_at=1781273651, file_counts=FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2), status='completed', vector_store_id='vs_4c6b1086-c038-431a-932b-c513cfadd9c1', object='vector_store.files_batch')


In [48]:
files = client.vector_stores.file_batches.list_files(
    vector_store_id=vector_store_id,
    batch_id=batch.id,
)
for f in files:
    print(f.id)

file-879c85a415534667a0b595b32175de90
file-b4fc6750d8304dc798ab0d317c332c4a


In [49]:
## creating a batch to verify cancel
batch = client.vector_stores.file_batches.create(
    vector_store_id=vector_store_id,
    file_ids=[file_info1.id, file_info2.id],
)
print(batch)

batch = client.vector_stores.file_batches.cancel(
    vector_store_id=vector_store_id,
    batch_id=batch.id
)
print(batch)

VectorStoreFileBatches(id='batch_754c5bb2-680e-46f6-8834-97c0a1430127', created_at=1781273681, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=2, total=2), status='in_progress', vector_store_id='vs_4c6b1086-c038-431a-932b-c513cfadd9c1', object='vector_store.files_batch')
VectorStoreFileBatches(id='batch_754c5bb2-680e-46f6-8834-97c0a1430127', created_at=1781273681, file_counts=FileCounts(cancelled=0, completed=0, failed=0, in_progress=2, total=2), status='cancelled', vector_store_id='vs_4c6b1086-c038-431a-932b-c513cfadd9c1', object='vector_store.files_batch')


In [50]:
vector_store = client.vector_stores.retrieve(vector_store_id)
print(vector_store)

VectorStore(id='vs_4c6b1086-c038-431a-932b-c513cfadd9c1', created_at=1781273638, file_counts=FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2), status='completed', expires_after=None, expires_at=None, last_active_at=1781273638, metadata={'provider_id': 'qdrant-remote', 'provider_vector_store_id': 'vs_4c6b1086-c038-431a-932b-c513cfadd9c1', 'embedding_model': 'vllm-embedding/granite-embeddings', 'embedding_dimension': '768'}, name='techmart_policy_store_remote_qdrant', object='vector_store', usage_bytes=0)


In [51]:
vector_store = client.vector_stores.update(
    vector_store_id,
    name="my-vector-store",
)
rich.print(vector_store)

VectorStore(
    id='vs_4c6b1086-c038-431a-932b-c513cfadd9c1',
    created_at=1781273638,
    file_counts=FileCounts(cancelled=0, completed=2, failed=0, in_progress=0, total=2),
    status='completed',
    expires_after=None,
    expires_at=None,
    last_active_at=1781273718,
    metadata={
        'provider_id': 'qdrant-remote',
        'provider_vector_store_id': 'vs_4c6b1086-c038-431a-932b-c513cfadd9c1',
        'embedding_model': 'vllm-embedding/granite-embeddings',
        'embedding_dimension': '768'
    },
    name='my-vector-store',
    object='vector_store',
    usage_bytes=0
)

In [52]:
rich.print(client.vector_stores.delete(vector_store_id))

VectorStoreDeleteResponse(
    id='vs_4c6b1086-c038-431a-932b-c513cfadd9c1',
    deleted=True,
    object='vector_store.deleted'
)

## Verify Vector store insert and Query

In [53]:
# Generate Embeddings
embedding_response = client.embeddings.create(
    input="Docker is a container platform.",
    model="vllm-embedding/granite-embeddings",
)

rich.print(embedding_response)
embedding = embedding_response.data[0].embedding

print(len(embedding))
print(embedding[:5])

CreateEmbeddingsResponse(
    data=[
        Data(
            embedding=[
                0.030732110142707825,
                -0.02075209468603134,
                -0.004593975376337767,
                -0.06178104504942894,
                -0.0025148054119199514,
                -0.013069067150354385,
                0.07065217196941376,
                -0.05227626860141754,
                -0.018217487260699272,
                0.04657340422272682,
                -0.01370271947234869,
                0.003722704015672207,
                0.015999706462025642,
                -0.00487119797617197,
                -0.0034256798680871725,
                -0.03944482281804085,
                -0.02265305072069168,
                0.0267717856913805,
                -0.01607891358435154,
                0.004574173595756292,
                0.01671256497502327,
                -0.016316533088684082,
                -0.016237325966358185,
                -0.05512770265340805,
                -0.018771933391690254,
                0.004613776691257954,
                0.06526613235473633,
                0.04007847234606743,
                -0.011484937742352486,
                0.0007326598279178143,
                -0.013465099968016148,
                -0.0004257347609382123,
                0.020435268059372902,
                0.01956399716436863,
                0.07191947102546692,
                -0.007841440849006176,
                -0.020276855677366257,
                -0.017821455374360085,
                0.04372197017073631,
                -0.021544160321354866,
                0.0011484938440844417,
                -0.010772080160677433,
                -0.0472070537507534,
                -0.030732110142707825,
                -0.01354430615901947,
                -0.040870536118745804,
                -0.006653343327343464,
                0.003920720424503088,
                0.005306833423674107,
                0.005188023671507835,
                0.0110096987336874,
                -0.04308832064270973,
                -0.007485011126846075,
                -0.02898956835269928,
                0.008554298430681229,
                -0.0020494672935456038,
                0.025662895292043686,
                -0.02898956835269928,
                -0.02455400489270687,
                -0.03992006182670593,
                -0.017504628747701645,
                0.024078765884041786,
                0.004732586443424225,
                -0.03310830518603325,
                -0.0002103921870002523,
                0.0141779575496912,
                -0.0031682588160037994,
                0.0035048862919211388,
                -0.04498927295207977,
                0.004039529711008072,
                0.034850846976041794,
                -0.016554152593016624,
                0.013623512350022793,
                -0.007960249669849873,
                0.0038217122200876474,
                -0.021227333694696426,
                -0.06336517632007599,
                -0.0157620869576931,
                -0.0057028657756745815,
                0.02455400489270687,
                0.007524614688009024,
                0.004059331491589546,
                -0.005940485280007124,
                -0.025346070528030396,
                0.004178141243755817,
                -0.0315241739153862,
                -0.007999853231012821,
                -0.05576135218143463,
                -0.011405731551349163,
                -0.06399882584810257,
                -0.04815753176808357,
                0.00962358620017767,
                0.04150418937206268,
                -0.009980015456676483,
                -0.01671256497502327,
                -0.005742468871176243,
                -0.04593975096940994,
                -0.025346070528030396,
                0.03089052252471447,
                -0.015603674575686455,
                0.030732110142707825,
                -0.026930199936032295,
                -0.007

768
[0.030732110142707825, -0.02075209468603134, -0.004593975376337767, -0.06178104504942894, -0.0025148054119199514]


In [54]:
# Create vector store with inline remote qdrant
vector_store = client.vector_stores.create(
    name="techmart_policy_store_remote_qdrant",
    extra_body={
        "embedding_model": embedding_model_id,
        "embedding_dimension": embedding_dimension,
        "provider_id": "qdrant-remote",
    },
)

vector_store_id = vector_store.id
print(f"Created vector store: {vector_store_id}")

Created vector store: vs_2458503c-22fc-40bf-9c8c-617f575c85fe


In [55]:
import time
import requests

now = int(time.time())

payload = {
    "vector_store_id": vector_store_id,
    "chunks": [
        {
            "chunk_id": "docker-1",
            "content": "Docker is a container platform.",
            "metadata": {},
            "chunk_metadata": {
                "chunk_id": "docker-1",
                "document_id": "docker-doc-1",
                "source": "manual",
                "created_timestamp": now,
                "updated_timestamp": now,
                "chunk_window": "0-100",
                "chunk_tokenizer": "nomic",
                "content_token_count": 5,
                "metadata_token_count": 0
            },
            "embedding": embedding,
            "embedding_model": "vllm-embedding/granite-embeddings",
            "embedding_dimension": 768
        }
    ]
}

r = requests.post(
    OGX_CONNECTION_URL+"/v1/vector-io/insert",
    json=payload
)

print(r.status_code)
print(r.text)

204



In [56]:
query_result = client.vector_io.query(
    vector_store_id=vector_store_id,
    query="docker",
    params={
        "mode": "vector", 
        "max_chunks": 1,
        
    },
)

print(query_result)

QueryChunksResponse(chunks=[Chunk(chunk_id='docker-1', chunk_metadata=ChunkChunkMetadata(chunk_id='docker-1', chunk_tokenizer='nomic', chunk_window='0-100', content_token_count=5, created_timestamp=1781273766, document_id='docker-doc-1', metadata_token_count=0, source='manual', updated_timestamp=1781273766), content='Docker is a container platform.', embedding=[0.030732110142707825, -0.02075209468603134, -0.004593975376337767, -0.06178104504942894, -0.002514805411919951, -0.013069067150354384, 0.07065217196941376, -0.05227626860141754, -0.018217487260699272, 0.04657340422272682, -0.01370271947234869, 0.003722704015672207, 0.015999706462025642, -0.00487119797617197, -0.0034256798680871725, -0.03944482281804085, -0.02265305072069168, 0.0267717856913805, -0.01607891358435154, 0.004574173595756292, 0.01671256497502327, -0.016316533088684082, -0.016237325966358185, -0.05512770265340805, -0.018771933391690258, 0.004613776691257954, 0.06526613235473633, 0.04007847234606743, -0.011484937742352

In [57]:
rich.print(query_result.scores)
rich.print(query_result.chunks[0].content)

[0.95378613]

Docker is a container platform.

In [58]:
query_result = client.vector_io.query(
    vector_store_id=vector_store_id,
    query="rain",
    params={
        "mode": "vector",
        "max_chunks": 1,
        
    },
)

rich.print(query_result.scores)
rich.print(query_result.chunks[0].content)

[0.72622406]

Docker is a container platform.

In [59]:
query_result = client.vector_io.query(
    vector_store_id=vector_store_id,
    query="Docker",
    params={
        "mode": "keyword",
        "max_chunks": 1,
        
    },
)

rich.print(query_result.scores)
# this will fails as earth is not inserted word
rich.print(query_result.chunks[0].content)

[]

IndexError: list index out of range

In [60]:
query_result = client.vector_io.query(
    vector_store_id=vector_store_id,
    query="rain",
    params={
        "mode": "keyword",
        "max_chunks": 1,
        
    },
)

rich.print(query_result)

QueryChunksResponse(chunks=[], scores=[])

In [61]:
query_result = client.vector_io.query(
    vector_store_id=vector_store_id,
    query="docker",
    params={
        "mode": "hybrid",
        "max_chunks": 1,
        
    },
)

rich.print(query_result.scores)
rich.print(query_result.chunks[0].content)

[]

IndexError: list index out of range

In [62]:
query_result = client.vector_io.query(
    vector_store_id=vector_store_id,
    query="rain",
    params={
        "mode": "hybrid",
        "max_chunks": 1,
        
    },
)

rich.print(query_result.scores)
rich.print(query_result.chunks[0].content)

[]

IndexError: list index out of range